<CENTER>
    <a href="http://opendata.atlas.cern" class="icons"><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/ATLASOD.gif?raw=1" style="width:50%"></a>
</CENTER>

# Kinematic Comparison of SM Higgs boson Decays and WW-Diboson Production
In this notebook, we'll look at how we can use PyROOT, a Python port of CERN's ROOT framework, to build a physics analysis! We'll do this using a simple analysis comparing the kinematic properties of events from two processes: the [Standard Model (SM)](https://cds.cern.ch/record/2759492/files/Standard%20Model%20-%20ATLAS%20Physics%20Cheat%20Sheet.pdf) Higgs boson decaying into two W bosons ($H \rightarrow WW^*$), and WW-diboson production ($qq\to WW$). The goal is to explore and visualize the differences in these processes, which can help in distinguishing between signal $H \rightarrow WW^*$ and background (WW-diboson) events in a particle physics analysis. Let's get started!

### Running a Jupyter notebook
A Jupyter notebook consists of cell blocks, each containing lines of Python code. Each cell can be run independently of each other, yielding respective outputs below the cells. Conventionally,cells are run in order from top to bottom.

- To run the whole notebook, in the top menu click Cell $\to$ Run All.
- To propagate a change you've made to a piece of code, click Cell $\to$ Run All Below.
- You can also run a single code cell, by clicking Cell $\to$ Run Cells, or using the keyboard shortcut Shift+Enter.

For more information, refer to [How To Use Jupyter Notebooks](https://www.codecademy.com/article/how-to-use-jupyter-notebooks).

## Higgs boson to WW Decay:

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/95dcff97ff8f0ae01e3aaf99f5285a5734a203cb/images/fig_01a.png?raw=1" style="width:30%"></CENTER>

The process we will be looking to identify in this analysis (called the *signal* process) is the decay of the Higgs boson into two W bosons $H \rightarrow WW^*$. According to the Standard Model, the Higgs boson arises from excitations in the Higgs Field during high energy events, temporarily creating a new particle as a result. The Higgs boson has a very short lifetime, and when it decays, it can produce a pair of W bosons, which are heavy bosons that mediate part of the weak force.

The [Feynman diagram](https://cds.cern.ch/record/2759490/files/Feynman%20Diagrams%20-%20ATLAS%20Cheat%20Sheet.pdf) above represents an example of this process. In this decay mode, two gluons fuse to form a quark loop which causes a Higgs boson to form. Usually, this loop is dominated by top quarks, as they are the heaviest quark flavor and cause a mass/energy fluctuation large enough to create a Higgs boson. The produced Higgs boson then decays into two W bosons. As the Higgs boson mass is less than twice the mass of the W boson, one of the produced W bosons will be considered a virtual "off-shell" boson, denoted by $W^*$. These W bosons can further decay into lepton-neutrino pairs and quarks. These final states can be detected directly with detectors like ATLAS, and kinematic properties of the events can be used to identify Higgs boson decays.

## WW-Diboson Production:

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/95dcff97ff8f0ae01e3aaf99f5285a5734a203cb/images/fig1b.png?raw=1" style="width:26%"></CENTER>

The second process we will look into is the production of a pair of W bosons through Standard Model WW-diboson production. In proton-proton collisions (like those at the Large Hadron Collider), exchanges in quark flavor and other quark-quark interactions can result in a pair of W bosons being produced. Depicted in the Feynman diagram above is an example of this production, where a pair of different quarks exchange a virtual quark, and radiate two W bosons. This is known as a t-channel exchange, and is the primary way diboson events are produced in particle collisions.

This will be our *background* process, as the end products of this process look nearly identical to our signal process. The aim of an analysis will be to separate our [signal from our background](https://cds.cern.ch/record/2800577/files/Signal%20and%20Background%20Physics%20Cheat%20Sheet.pdf), and make sure we know that what we're looking at is signal!

## Process Kinematics

If both processes have the same final state, how wil we separate our signal from the backgrounds? The easiest way to do this is using kinematic properties of the processes, and comparing slight differences in end-states from how each process occurs. In our analysis, we will do this using missing energy and jets. 

Sometimes when a W boson decays, it creates neutrinos, which cannot be detected directly and introduce missing energy. In our signal sample, both W bosons originate from the decay of a Higgs boson, which has an [invariant mass](https://cds.cern.ch/record/2919871/files/4-vectors%20and%20invariant%20mass%20cheat%20sheet.pdf) around 125 GeV. Since the products of the Higgs boson decay will have the same total energy as the Higgs boson, the product energies will be directly correlated to the Higgs boson mass. W bosons decay into a lepton-neutrino pair 33% of the time, meaning that events with around 30-60 GeV of missing energy suggest we are seeing between 1 and 2 lepton-neutrino pairs created as a product of $H \to WW^*$ decay. This is a mass peak unique to the Higgs Boson, giving us a quality we can use to differentiate the processes.

Another kinematic difference arises in the jets found in each [hard-scatter process](https://cds.cern.ch/record/2951043/files/Inside%20a%20Collision.pdf). Gluon-gluon fusion produces no jets, so the only way jets can appear in these processes are through initial state radiation (ISR) prior to the hard scatter process. Further, the energy scale that our signal process occurs at is fixed around 125 GeV, in contrast to the WW-diboson production which can occur at any energy scale. This makes the WW-diboson background process stand out against our signal process, as even though our background produces no jets on its own, it is often grouped in with other high energy jet-producing processes due to its varying energy scale. This means that in theory, our signal process should generally present 0 jets, while our background process will contain contamination from other similar jet-producing processes. 

## Our Analysis

We begin by importing CERN's `ROOT` library, a tool widely used in High Energy Physics (HEP) for data analysis, visualization, and handling large datasets. Since we are running this in a python kernel importing the `ROOT` library is synonymous with importing PyROOT. The `%jsroot on` command then enables JavaScript-based interactive plots in the notebook environment, allowing us to visualize the outputs directly in the browser.

In [ ]:
import ROOT
%jsroot on

Next, we'll load our data by importing and using the `atlasopenmagic` package, allowing us to stream data and [Monte Carlo](https://cds.cern.ch/record/2922919/files/Monte%20Carlo%20Simulation.pdf) files directly from the CERN Open Data portal without downloading them individually. 

In [ ]:
# We'll install and import the Atlasopenmagic package
%pip install atlasopenmagic
import atlasopenmagic as atom
# And let's plan to use the latest release of Open Data for Outreach and Education
atom.set_release('2025e-13tev-beta')

Here we've pre-selected a group of signal and background simulated samples. These are ideal computer-generated recreations of the actual physical processes created usign the Monte Carlo simulation method. We use these Monte Carlo datasets as they allow us more control over visualizing our data, as we can use pre-identified variables and the truth record to see exactly what we want to see. These datasets will be stored in `.root` files, and we use the `TChain()` method and `atlasopenmagic` package to open these files and extract the event trees. We also check the number of events in each tree to get an idea of how much data we have to work with, and if our import was successful. 

In [ ]:
samples_list = {
    r'Background Diboson':      {'dids': [700601, 700602, 700603, 700604], 'color': 'red'},
    r'Signal $H #rightarrow WW$':     {'dids': [345948], 'color': 'blue'}
}

# In case you are working on a system that doesn't have xrootd, you can use the https protocol to read 
# files with the following line:
# mc_samples = atom.build_dataset(samples_list, skim='2to4lep', protocol='https', cache = True)

mc_samples = atom.build_dataset(samples_list, skim='2to4lep', protocol='root')

print(f"MC Files Loaded: {len(mc_samples)}")

signal_chain = ROOT.TChain("analysis")
background_chain = ROOT.TChain("analysis")

for name_sample in mc_samples:
    for url in mc_samples[name_sample]['list']:
        if 'Background' in name_sample:
            background_chain.Add(url)
        else:
            signal_chain.Add(url)

signal_entries = signal_chain.GetEntries()
print(f"Number of events in the H->WW dataset: {signal_entries}")

background_entries = background_chain.GetEntries()
print(f"Number of events in the diboson dataset: {background_entries}")

After loading the data, we set up a `TCanvas`, which acts as the drawing area for the plots we will generate. The parameters for the canvas specify its title and size.

In [ ]:
c = ROOT.TCanvas("testCanvas","a first way to plot a variable",800,600)

In this analysis, we are interested in comparing two key kinematic variables: the missing transverse energy `met` and the number of jets `jet_n` in each event. To do this, we define histograms for both variables for both the background and signal datasets. Our "h" histogram will represent the `met` centered analysis, while our "h2" histogram will represent our `jet_n` analysis.

In [ ]:
h_bgs = ROOT.TH1F("h_bgs","Missing transverse energy in WW final states; MET [GeV]; Probability Density",20,0,200)
h2_bgs = ROOT.TH1F("h2_bgs","Number of Jets in WW final states; jet_n; Probability Density",10,0,10)

h_sig = ROOT.TH1F("h_sig","Missing transverse energy in WW final states; MET [GeV]; Probability Density",20,0,200)
h2_sig = ROOT.TH1F("h2_sig","Number of Jets in WW final states; jet_n; Probability Density",10,0,10)

Now we fill these histograms with data. The following code will loop over all the events in both the background and signal datasets. For each event, we load the missing transverse energy values and the number of jets into their respective histograms. This is the main analysis step of the notebook, so allow 2-5 minutes for the following cell to complete. Once the background and signal processes are loaded into their histograms, the cell will output a message letting you know things are complete!

In [ ]:
for event in background_chain:
    h_bgs.Fill((background_chain.met))
    h2_bgs.Fill(background_chain.jet_n)

print("Background processed.")

for event in signal_chain:
    h_sig.Fill((signal_chain.met))
    h2_sig.Fill(signal_chain.jet_n)
        
print("Signal processed.")

Once the histograms are filled, we normalize them. This ensures that the total area under each histogram equals 1, allowing us to compare the shapes of the distributions rather than just the number of events. This step is important for making a meaningful comparison between the signal and background.

In [ ]:
scale_bgs = h_bgs.Integral()
h_bgs.Scale(1/scale_bgs)

scale_sig = h_sig.Integral()
h_sig.Scale(1/scale_sig)

## Plotting

### MET Plot

Finally, we'll define the style of the histograms and draw them onto our canvases. We'll apply different visual choices such as setting different colors and fill styles for the signal and background bins. This helps to visually differentiate between them. We also create a legend that labels the signal and background histograms, making the plot easier to interpret. After this cell runs, we'll be able to interperet our first set of data: the `met` kinematic difference between our signal and background models. 

In [ ]:
h_bgs.SetFillStyle(3001)
h_bgs.SetFillColor(4)
h_bgs.SetLineColor(4)

h_sig.SetFillStyle(3003)
h_sig.SetFillColor(2)
h_sig.SetLineColor(2)

legend=ROOT.TLegend(0.5,0.7,0.9,0.9)
legend.AddEntry(h_bgs,"Background (Diboson WW) ","l")
legend.AddEntry(h_sig,"Signal (H #rightarrow WW)","l")

h_sig.SetStats(0)
h_bgs.SetStats(0)

h_sig.Draw("hist")
h_bgs.Draw("histsame")
legend.Draw()
c.Draw()

# If you want a copy of your plot locally, uncomment the following line
# c.SaveAs("HWW_MET_comparison.pdf") # pdf image

This plot compares the missing transverse energy distribution between the WW-diboson simulated background and the $H \to WW$ simulated signal. 

Immediately, we can see that the models both have a very similar distribution. Both show a sharp rise and peak at the 40-50 GeV bin. This is expected, as the majority of neutrinos produced will be around half of the rest mass of the W boson, near 40 GeV. However, we can also notice equally significant differences in the heights of either peak. 

The WW-diboson distribution, in which the hard process occurs at all energy scales, has a broader distribution of `met` as the W bosons can have any amount of transverse momentum and boost their resulting neutrinos. Conversely, the $H \to WW$ process peaks more strongly in the 40-50 GeV area, and is more tightly concentrated around its peak, with a much smaller tail. This is because the Higgs Boson intermediate step locks the W boson masses to sum to 125 GeV, forcing the distribution of `met` more tightly between one and two quarters of the higgs mass.

### Jet Multiplicity Plot

Next, we'll repeat the prior steps of normalization, styling, and plotting, but for the jet distribution! In this case, we'll run the same code, using the "h2" histogram instead. 

In [ ]:
scale2_bgs = h2_bgs.Integral()
h2_bgs.Scale(1/scale2_bgs)

scale2_sig = h2_sig.Integral()
h2_sig.Scale(1/scale2_sig)

h2_bgs.SetFillStyle(3001)
h2_bgs.SetFillColor(4)
h2_bgs.SetLineColor(4)

h2_sig.SetFillStyle(3003)
h2_sig.SetFillColor(2)
h2_sig.SetLineColor(2)

legend=ROOT.TLegend(0.5,0.7,0.9,0.9)
legend.AddEntry(h2_bgs,"Background (Diboson WW) ","l")
legend.AddEntry(h2_sig,"Signal (H #rightarrow WW)","l")

h2_sig.SetStats(0)
h2_bgs.SetStats(0)
h2_sig.Draw("hist")
h2_bgs.Draw("histsame")
legend.Draw()
c.Draw()

# If you want a copy of your plot locally, uncomment the following line
# c.SaveAs("HWW_jet_n_comparison.pdf") # pdf image

This plot compares the jet multiplicity between the WW-diboson simulated background and the $H \to WW$ simulated signal. 

In WW-Diboson production, the broad energy scales that can cause the process to occur allow incoming gluons to enter with very high momenta. This high initial-state momenta allow for incoming gluons to more readily emit high-energy hadronic jets while still forming W bosons. This leads to a spread out distribution across the 0-, 1- and 2-jet bins, as incoming gluons at higher energies contribute jets to the process count. 

In $H \to WW$, the incoming gluons will be of lower energy, as they must sum to ~125 GeV to produce a Higgs Boson. This means that each incoming gluon will have much less energy to eject a detectable jet, and we will detect fewer jets from these interactions. This means that a jet distribution of $H \to WW$ will be primarily concentrated in the 0-jet bin, with a smaller 1-jet bin, and very few for 2+ jet bins.

## Summary

In this notebook, we used PyROOT to perform a kinematic comparison between two important Standard Model processes: the Higgs boson decay to two W bosons ($H \rightarrow WW^*$) and the WW-diboson background. By analyzing the missing transverse energy (MET) and the number of jets in these events, we explored how the signal and background distributions differ, providing insight into how these variables can be used to separate signal from background in a typical particle physics analysis.

We hope you enjoyed this notebook, and for an extra challenge:

- Try another comparison using the opening angles ($\delta\phi$) between the signal and background samples. What do you notice? What could this suggest about the diboson production channel?
- Think about how you could remake this analysis using other Higgs/Non-Higgs processes. How could we compare $H \to ZZ^*$? Or $H \to b\bar{b}$ and $t \to Wb$?
- Try adapting your favorite analysis from Open Data or your own work to PyROOT! Think about the major differences or obstacles you may run into and how you could adapt your analysis to them,
<div class="alert alert-block alert-info">
We welcome your feedback on this notebook or any of our other materials! Please <a href="https://forms.gle/zKBqS1opAHHemv9U7">fill out this survey</a> to let us know how we're doing, and you can enter a raffle to win some <a href="https://atlas-secretariat.web.cern.ch/merchandise">ATLAS merchandise</a>!
</div>